# Main Script as Jupyter Notebook

This notebook is a conversion of the `main.py` script, designed for easier debugging by running cell by cell.

In [ ]:
# Imports and Initial Setup
import os 
import argparse # Kept for reference, but parameters will be set manually
import configparser
import pandas as pd
import torch
import pytorch_lightning as pl


from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objs as go
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from pynas.core.population import Population
from datasets.RawVessels.loader import RawVesselsDataModule, RawVesselsDataset

import numpy as np
import cv2
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import hilbert
from skimage import io
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import blob_log, peak_local_max
# Pandas display option
pd.set_option('display.max_colwidth', None)
# PyTorch Lightning seed and precision
pl.seed_everything(seed=42, workers=True) # Example seed, change as needed
torch.set_float32_matmul_precision("medium")

# Utils

In [ ]:
import numpy as np

def extract_dominant_phase_components(phase_matrix: np.ndarray, N: int):
    """
    Extract the N dominant frequency components from a wrapped SAR phase matrix using FFT.

    Args:
        phase_matrix (np.ndarray): 2D wrapped phase matrix (real-valued, in [-π, π]).
        N (int): Number of dominant frequency components to extract.

    Returns:
        List[Tuple[Tuple[int, int], complex, np.ndarray]]: List of N tuples:
            - ((fx, fy), amplitude, spatial_pattern) where fx, fy are frequency indices,
              amplitude is the complex FFT coefficient, and spatial_pattern is the
              reconstructed phase matrix for that component.
    """
    if phase_matrix.ndim != 2:
        raise ValueError("Input phase_matrix must be 2D.")

    # Compute the 2D FFT
    fft_matrix = np.fft.fftshift(np.fft.fft2(phase_matrix))

    # Get the flattened index list sorted by magnitude (descending)
    magnitude = np.abs(fft_matrix)
    flat_indices = np.argsort(magnitude.ravel())[::-1]

    # Map flat indices to 2D frequency indices
    dominant_components = []
    rows, cols = phase_matrix.shape
    center_r, center_c = rows // 2, cols // 2

    for idx in flat_indices[:N]:
        fy, fx = np.unravel_index(idx, fft_matrix.shape)
        amplitude = fft_matrix[fy, fx]
        # Shift to frequency coordinates relative to center (0 frequency at center)
        fx_shifted = fx - center_c
        fy_shifted = fy - center_r

        # Create a 2D array with only this frequency component (in frequency domain)
        fft_component = np.zeros_like(fft_matrix, dtype=complex)
        fft_component[fy, fx] = amplitude
        # Inverse FFT to get spatial pattern
        spatial_pattern = np.fft.ifft2(np.fft.ifftshift(fft_component)).real

        dominant_components.append(((fx_shifted, fy_shifted), amplitude, spatial_pattern))

    return dominant_components

# Data Loading

In [ ]:
# ----- Load the dataset ------
root_dir_datamodule = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined'
# Ensure the root directory exists
if not Path(root_dir_datamodule).exists():
    raise FileNotFoundError(f"Root directory {root_dir_datamodule} does not exist.")
# Load image and mask paths
root_dir_datamodule = Path(root_dir_datamodule)
if not (root_dir_datamodule / 'inputs').exists() or not (root_dir_datamodule / 'masks').exists():
    raise FileNotFoundError(f"Expected directories 'inputs' and 'masks' not found in {root_dir_datamodule}.")

image_paths = [x for x in (root_dir_datamodule / 'inputs').glob('*.pkl')]
mask_paths = [x for x in (root_dir_datamodule / 'masks').glob('*.pkl')]


loader = RawVesselsDataset(image_paths, 
                           mask_paths, 
                           transform=None)


In [ ]:
# ----- Load the dataset ------
# Ensure the dataset is loaded correctly
# Try loading and inspecting a sample
rand_idx = 1  # Change this index to load different samples
sample = loader[rand_idx]
img, mask = sample 


Re, Im = img  # Real and Imaginary parts of the complex image
# Make Amplitude and Phase
amp = np.abs(Re + 1j * Im)  # Amplitude
phase = np.angle(Re + 1j * Im)  # Phase

# Compute mean and std for phase
mean = np.mean(phase)
std = np.std(phase)
vmin = mean - 1.5 * std
vmax = mean + 1.5 * std

fig, axs = plt.subplots(1, 3, figsize=(14, 7), dpi=140)

axs[0].imshow(amp, cmap='gray')
axs[0].set_title('Amplitude')
axs[0].axis('off')

# Increased contrast for phase
axs[1].imshow(phase, cmap='inferno', vmin=vmin, vmax=vmax)
axs[1].set_title('Phase (Increased Contrast)')
axs[1].axis('off')

axs[2].imshow(mask, cmap='gray')
axs[2].set_title('Ground Truth Mask')
axs[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
""" DEPRECATED: This section is not used in the final code
phase_components = extract_dominant_phase_components(phase, N=35)

# Plot all 35 dominant phase components (spatial patterns)
n_components = 35
n_rows, n_cols = 7, 5  # 7x5=35

fig, axs = plt.subplots(n_rows, n_cols, figsize=(22, 18), dpi=140)

for i, ((fx, fy), amplitude, spatial_pattern) in enumerate(phase_components[:n_components]):
    ax = axs[i // n_cols, i % n_cols]
    ax.imshow(spatial_pattern, cmap='inferno')
    ax.set_title(f'({fx}, {fy})')
    ax.axis('off')

plt.suptitle('Top 35 Dominant Phase Components (Spatial Patterns)')
plt.tight_layout()
plt.show()"""

# Fringe Method A

In [ ]:



def hilbert_analytic_signal(image):
    analytic_signal = hilbert(image, axis=0) + 1j * hilbert(image, axis=1)
    return np.angle(analytic_signal)

def bandpass_filter(phase, r_in=15, r_out=60):
    h, w = phase.shape
    f = np.fft.fftshift(np.fft.fft2(phase))
    H = np.zeros_like(phase)
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    H[(dist >= r_in) & (dist <= r_out)] = 1
    f_filtered = f * H
    return np.angle(np.fft.ifft2(np.fft.ifftshift(f_filtered)))

def normalize_and_equalize(phase_image):
    norm = ((phase_image - phase_image.min()) / (phase_image.max() - phase_image.min()) * 255).astype(np.uint8)
    return cv2.equalizeHist(norm)

def refine_enhancement(image):
    tv_denoised = denoise_tv_chambolle(image.astype(np.float32), weight=0.2)
    return median_filter(tv_denoised, size=5)

def estimate_scatterer_location(enhanced_image, initial_guess=None):
    blobs = blob_log(enhanced_image, min_sigma=3, max_sigma=20, num_sigma=10, threshold=0.02)
    blobs = blobs[np.argsort(-blobs[:, 2])]
    if initial_guess is None:
        return int(blobs[0][0]), int(blobs[0][1])
    else:
        local_peaks = peak_local_max(enhanced_image, min_distance=10, threshold_abs=0.1)
        dists = np.linalg.norm(local_peaks - np.array(initial_guess), axis=1)
        return tuple(local_peaks[np.argmin(dists)])




# --------------- Main Processing Pipeline ---------------

smoothed = gaussian_filter(img, sigma=2)
phase = hilbert_analytic_signal(smoothed)
filtered_phase = bandpass_filter(phase)
enhanced = normalize_and_equalize(filtered_phase)
cleaned = refine_enhancement(enhanced)
init_y, init_x = estimate_scatterer_location(cleaned)
refined_y, refined_x = estimate_scatterer_location(cleaned, initial_guess=(init_y, init_x))

# Plot results
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray')
plt.scatter(refined_x, refined_y, c='lime', s=100, marker='x', label='Refined Scatterer')
plt.title("Refined Scatterer Location")
plt.legend()
plt.show()



📌 Use Cases
	•	Fringe pattern orientation analysis.
	•	Directional filtering (e.g., Hough-style for SAR fringes).
	•	Initial phase ramp estimation for correction in InSAR.

Let me know if you want the inverse Radon (reconstruction) or directional filtering based on Radon peaks.

In [ ]:
import numpy as np
from skimage.transform import radon

def compute_radon_transform(phase_matrix: np.ndarray, theta: np.ndarray = None):
    """
    Compute the Radon transform of a 2D wrapped phase matrix.

    Args:
        phase_matrix (np.ndarray): 2D input phase matrix (real-valued).
        theta (np.ndarray, optional): Array of projection angles (degrees). 
            Defaults to np.linspace(0., 180., max(phase_matrix.shape), endpoint=False).

    Returns:
        radon_image (np.ndarray): 2D Radon transform (ρ × θ).
        theta (np.ndarray): Array of angles used.
    """
    if theta is None:
        theta = np.linspace(0., 180., max(phase_matrix.shape), endpoint=False)

    radon_image = radon(phase_matrix, theta=theta, circle=False)
    return radon_image, theta

In [ ]:
# Plot only the real and imaginary parts
fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=140)

axs[0].imshow(Re, cmap='inferno')
axs[0].set_title('Real Part')
axs[0].axis('off')

axs[1].imshow(Im, cmap='inferno')
axs[1].set_title('Imaginary Part')
axs[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:


# Plot the difference between the real and imaginary parts
diff = Re - Im

plt.figure(figsize=(6, 5), dpi=140)
plt.imshow(diff, cmap='inferno')
plt.title('Difference: Real - Imaginary')
plt.axis('off')
plt.colorbar()
plt.show()

# 3D surface plot of the normalized difference
diff_norm = (diff - np.mean(diff)) / np.std(diff)  # z-score normalization


X = np.arange(diff_norm.shape[1])
Y = np.arange(diff_norm.shape[0])
X, Y = np.meshgrid(X, Y)


fig = go.Figure(data=[go.Surface(z=diff_norm, x=X, y=Y, colorscale='RdBu', cmin=-1, cmax=1, colorbar=dict(title='z-score'))])
fig.update_layout(
    title='3D Surface: Normalized Difference (z-score)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='z-score'
    ),
    autosize=False,
    width=800,
    height=600,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig.show()




In [ ]:
# 3D surface plot of the phase (z-score normalized)
phase_norm = (phase - np.mean(phase)) / np.std(phase)

fig_phase = go.Figure(data=[go.Surface(z=phase_norm, x=X, y=Y, colorscale='Viridis', colorbar=dict(title='z-score'))])
fig_phase.update_layout(
    title='3D Surface: Normalized Phase (z-score)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='z-score'
    ),
    autosize=False,
    width=1800,
    height=1600,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig_phase.show()

In [ ]:
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))
print("Batch keys:", batch.keys() if hasattr(batch, 'keys') else type(batch))
print("Batch shapes:")
for k, v in batch.items():
    print(f"{k}: {v.shape}" if hasattr(v, 'shape') else f"{k}: {type(v)}")

In [ ]:
# Configuration Parameters (Replaces argparse and config.ini for notebook usage)

# Simulating command line arguments (set manually for debugging)
class Args:
    def __init__(self):
        self.gen = None  # Or specify a generation number, e.g., 1, 2, ...
        # self.config = 'config.ini' # config.ini will be read effectively by hardcoding params below

args = Args()

# Simulating config.ini parameters (set manually for debugging)
# These values would typically be read from config.ini
config_params = {
    'Computation': {
        'seed': 42  # Default seed
    },
    'NAS': {
        'max_layers': 10
    },
    'GA': {
        'max_iterations': 50,
        'population_size': 20,
        'mating_pool_cutoff': 0.5,
        'mutation_probability': 0.2,
        'epochs': 10,
        'batch_size': 4, # Note: dm also has batch_size, ensure consistency or decide which one to use
        'n_random': 5,
        'k_best': 5
    }
}

# Apply seed and precision from config
pl.seed_everything(seed=config_params['Computation']['seed'], workers=True)  # For reproducibility
torch.set_float32_matmul_precision("medium")  # to make lightning happy

print("Configuration parameters set.")
print(f"Generation to load: {args.gen}")
print(f"Seed: {config_params['Computation']['seed']}")

In [ ]:
# Main Function Definition
def main_notebook(notebook_args, notebook_config, datamodule):
    # Seed and precision are set in the cell above based on notebook_config

    # Model parameters from notebook_config
    max_layers = int(notebook_config['NAS']['max_layers'])
    max_gen = int(notebook_config['GA']['max_iterations'])
    n_individuals = int(notebook_config['GA']['population_size'])
    mating_pool_cutoff = float(notebook_config['GA']['mating_pool_cutoff'])
    mutation_probability = float(notebook_config['GA']['mutation_probability'])
    epochs = int(notebook_config['GA']['epochs'])
    batch_size = int(notebook_config['GA']['batch_size']) # Ensure this aligns with dm or is used consistently
    n_random = int(notebook_config['GA']['n_random'])
    k_best = int(notebook_config['GA']['k_best'])
    
    print("Initializing Population...")
    # Define population
    pop = Population(n_individuals=n_individuals, max_layers=max_layers, dm=datamodule, max_parameters=400_000)
    
    if notebook_args.gen is not None:
        print(f"Loading generation: {notebook_args.gen}")
        pop.load_generation(notebook_args.gen)
    else:
        print("Performing initial poll for population...")
        pop.initial_poll()

    print(f"Starting training for {max_gen} generations...")
    for gen_idx in range(max_gen):
        print(f"--- Generation {gen_idx + 1}/{max_gen} ---")
        print("Training generation...")
        pop.train_generation(task='classification', lr=0.001, epochs=epochs, batch_size=batch_size)
        print("Evolving population...")
        pop.evolve(mating_pool_cutoff=mating_pool_cutoff, mutation_probability=mutation_probability, k_best=k_best, n_random=n_random)
        print(f"--- End of Generation {gen_idx + 1} ---")
        
    print("Process finished.")

print("Main function 'main_notebook' defined.")

In [ ]:
# Call the Main Function
# This replaces the if __name__ == '__main__': block

# Ensure 'dm' (DataModule), 'args' (simulated command-line args), 
# and 'config_params' (simulated config.ini) are defined from previous cells.

print("Running main_notebook function...")
main_notebook(args, config_params, dm)
print("Notebook execution complete.")